In [1]:
import os
import pandas as pd
import numpy as np
import s3fs
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
S3_BUCKET_CURATED = os.getenv("S3_BUCKET_CURATED", "curated")
MINIO_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
MINIO_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin123")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio:9000")
MINIO_SECURE = os.getenv("MINIO_SECURE", "false").lower() == "true"

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB = os.getenv("POSTGRES_DB", "oil_pipeline")
POSTGRES_USER = os.getenv("POSTGRES_USER", "oil_user")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "oil_password")

In [3]:
def get_s3_fs():
    protocol = "https" if MINIO_SECURE else "http"
    return s3fs.S3FileSystem(
        key=MINIO_ACCESS_KEY,
        secret=MINIO_SECRET_KEY,
        client_kwargs={"endpoint_url": f"{protocol}://{MINIO_ENDPOINT}"},
    )

def get_pg_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
        f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
    )

In [4]:
fs = get_s3_fs()

production_clean = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/clean/production/production_clean.parquet",
    filesystem=fs
)

well_telemetry_clean = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/clean/well_telemetry/well_telemetry_clean.parquet",
    filesystem=fs
)

mart_production_daily = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/marts/mart_production_daily.parquet",
    filesystem=fs
)

mart_well_kpi = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/marts/mart_well_kpi.parquet",
    filesystem=fs
)

In [5]:
production_clean["date"] = pd.to_datetime(production_clean["date"])
well_telemetry_clean["timestamp"] = pd.to_datetime(well_telemetry_clean["timestamp"])
mart_production_daily["date"] = pd.to_datetime(mart_production_daily["date"])

In [6]:
production_clean.head(), mart_production_daily.head(), mart_well_kpi.head()

(   production_id  well_id       date  oil_ton   gas_m3  water_m3  energy_kwh  \
 0              1        1 2025-10-01    212.4  55200.0     182.3      7450.0   
 1             31        2 2025-10-01    186.1  49800.0     162.0      6800.0   
 2             61        3 2025-10-01    121.8  40120.0     121.5      5280.0   
 3             91        4 2025-10-01      0.0      0.0       0.0         0.0   
 4            121        5 2025-10-01    197.2  52050.0     165.3      7200.0   
 
    downtime_hours  temperature  pressure  
 0             0.5         88.1     120.4  
 1             0.8         84.5     115.4  
 2             2.0         79.4     107.1  
 3            24.0          NaN       NaN  
 4             0.4         86.5     118.8  ,
         date  total_oil_ton  total_gas_m3  total_water_m3  total_energy_kwh  \
 0 2025-10-01          717.5      197170.0           631.1           26730.0   
 1 2025-10-02          717.3      197200.0           630.5           26735.0   
 2 2025

## Общая добыча по дням

На основе очищенной таблицы production построена витрина mart_production_daily, содержащая агрегаты по суткам: суммарную добычу нефти, газа, воды, энергопотребление и простой.

In [7]:
daily_summary = mart_production_daily.copy()

daily_summary[[
    "date",
    "total_oil_ton",
    "total_gas_m3",
    "total_water_m3",
    "total_energy_kwh",
    "total_downtime_hours",
    "downtime_pct"
]].head(10)

,date,total_oil_ton,total_gas_m3,total_water_m3,total_energy_kwh,total_downtime_hours,downtime_pct
0,2025-10-01,717.5,197170.0,631.1,26730.0,27.7,23.083333
1,2025-10-02,717.3,197200.0,630.5,26735.0,28.0,23.333333
2,2025-10-03,719.2,197350.0,630.8,26770.0,27.8,23.166667
3,2025-10-04,722.9,197810.0,627.4,26875.0,26.7,22.250000
4,2025-10-05,721.0,197620.0,630.1,26810.0,27.5,22.916667
5,2025-10-06,718.6,197350.0,632.6,26770.0,28.1,23.416667
6,2025-10-07,714.1,196800.0,635.5,26660.0,29.4,24.500000
7,2025-10-08,716.4,197090.0,631.9,26725.0,27.6,23.000000
8,2025-10-09,721.7,197700.0,628.6,26840.0,27.0,22.500000
9,2025-10-10,718.5,197330.0,631.3,26760.0,27.9,23.250000


In [8]:
overall_metrics = pd.DataFrame({
    "metric": [
        "total_oil_ton",
        "avg_daily_oil_ton",
        "max_daily_oil_ton",
        "min_daily_oil_ton",
        "avg_downtime_pct",
        "avg_temperature",
        "avg_pressure"
    ],
    "value": [
        daily_summary["total_oil_ton"].sum(),
        daily_summary["total_oil_ton"].mean(),
        daily_summary["total_oil_ton"].max(),
        daily_summary["total_oil_ton"].min(),
        daily_summary["downtime_pct"].mean(),
        daily_summary["avg_temperature"].mean(),
        daily_summary["avg_pressure"].mean()
    ]
})

overall_metrics

,metric,value
0,total_oil_ton,21572.800000
1,avg_daily_oil_ton,719.093333
2,max_daily_oil_ton,724.300000
3,min_daily_oil_ton,714.100000
4,avg_downtime_pct,22.994444
5,avg_temperature,84.731667
6,avg_pressure,115.609167


## KPI по скважинам

Для каждой скважины рассчитаны:
- суммарная добыча;
- средний суточный дебит;
- процент простоя;
- средняя температура;
- среднее давление;
- средние телеметрические показатели.

In [9]:
well_kpi = mart_well_kpi.copy()

well_kpi[[
    "well_id",
    "total_oil_ton",
    "avg_oil_ton",
    "downtime_pct",
    "avg_prod_temperature",
    "avg_prod_pressure",
    "avg_telemetry_temperature",
    "avg_telemetry_pressure"
]].sort_values("avg_oil_ton", ascending=False)

,well_id,total_oil_ton,avg_oil_ton,downtime_pct,avg_prod_temperature,avg_prod_pressure,avg_telemetry_temperature,avg_telemetry_pressure
0,1,6394.5,213.150000,2.013889,88.206667,120.543333,88.027264,122.016986
1,5,5952.9,198.430000,1.819444,86.800000,119.426667,86.521034,119.422141
2,2,5574.5,185.816667,3.027778,84.486667,115.306667,84.328472,115.412069
3,3,3650.9,121.696667,8.111111,79.433333,107.160000,NaN,NaN
4,4,0.0,0.000000,100.000000,NaN,NaN,NaN,NaN


In [10]:
top_wells = well_kpi.sort_values("avg_oil_ton", ascending=False).head(5)
worst_wells = well_kpi.sort_values("avg_oil_ton", ascending=True).head(5)

top_wells[["well_id", "avg_oil_ton", "downtime_pct"]], worst_wells[["well_id", "avg_oil_ton", "downtime_pct"]]

(   well_id  avg_oil_ton  downtime_pct
 0        1   213.150000      2.013889
 1        5   198.430000      1.819444
 2        2   185.816667      3.027778
 3        3   121.696667      8.111111
 4        4     0.000000    100.000000,
    well_id  avg_oil_ton  downtime_pct
 4        4     0.000000    100.000000
 3        3   121.696667      8.111111
 2        2   185.816667      3.027778
 1        5   198.430000      1.819444
 0        1   213.150000      2.013889)

In [11]:
highest_downtime = well_kpi.sort_values("downtime_pct", ascending=False).head(5)
highest_downtime[["well_id", "avg_oil_ton", "downtime_pct", "total_downtime_hours"]]

,well_id,avg_oil_ton,downtime_pct,total_downtime_hours
4,4,0.000000,100.000000,720.0
3,3,121.696667,8.111111,58.4
2,2,185.816667,3.027778,21.8
0,1,213.150000,2.013889,14.5
1,5,198.430000,1.819444,13.1


In [12]:
corr_matrix = production_clean[["oil_ton", "pressure", "temperature", "downtime_hours"]].corr()
corr_matrix

,oil_ton,pressure,temperature,downtime_hours
oil_ton,1.000000,0.987107,0.985105,-0.938873
pressure,0.987107,1.000000,0.987276,-0.940295
temperature,0.985105,0.987276,1.000000,-0.890180
downtime_hours,-0.938873,-0.940295,-0.890180,1.000000


In [13]:
production_clean["pressure_bin"] = pd.cut(production_clean["pressure"], bins=5)

pressure_impact = (
    production_clean.groupby("pressure_bin", observed=False)
    .agg(
        avg_oil_ton=("oil_ton", "mean"),
        avg_temperature=("temperature", "mean"),
        rows=("oil_ton", "count")
    )
    .reset_index()
)

pressure_impact

,pressure_bin,avg_oil_ton,avg_temperature,rows
0,"(106.385, 109.42]",121.696667,79.433333,30
1,"(109.42, 112.44]",NaN,NaN,0
2,"(112.44, 115.46]",185.268421,84.605263,19
3,"(115.46, 118.48]",186.763636,84.281818,11
4,"(118.48, 121.5]",205.790000,87.503333,60


In [14]:
production_clean["temperature_bin"] = pd.cut(production_clean["temperature"], bins=5)

temperature_impact = (
    production_clean.groupby("temperature_bin", observed=False)
    .agg(
        avg_oil_ton=("oil_ton", "mean"),
        avg_pressure=("pressure", "mean"),
        rows=("oil_ton", "count")
    )
    .reset_index()
)

temperature_impact

,temperature_bin,avg_oil_ton,avg_pressure,rows
0,"(78.99, 81.02]",121.696667,107.160000,30
1,"(81.02, 83.04]",NaN,NaN,0
2,"(83.04, 85.06]",185.816667,115.306667,30
3,"(85.06, 87.08]",198.430000,119.426667,30
4,"(87.08, 89.1]",213.150000,120.543333,30


In [15]:
heatmap_source = production_clean.copy()

heatmap_source["pressure_bin"] = pd.cut(heatmap_source["pressure"], bins=6)
heatmap_source["oil_bin"] = pd.cut(heatmap_source["oil_ton"], bins=6)

pressure_oil_heatmap = (
    heatmap_source.groupby(["pressure_bin", "oil_bin"], observed=False)
    .size()
    .reset_index(name="records_count")
)

pressure_oil_heatmap.head(20)

,pressure_bin,oil_bin,records_count
0,"(106.385, 108.917]","(-0.215, 35.867]",0
1,"(106.385, 108.917]","(35.867, 71.733]",0
2,"(106.385, 108.917]","(71.733, 107.6]",0
3,"(106.385, 108.917]","(107.6, 143.467]",30
4,"(106.385, 108.917]","(143.467, 179.333]",0
5,"(106.385, 108.917]","(179.333, 215.2]",0
6,"(108.917, 111.433]","(-0.215, 35.867]",0
7,"(108.917, 111.433]","(35.867, 71.733]",0
8,"(108.917, 111.433]","(71.733, 107.6]",0
9,"(108.917, 111.433]","(107.6, 143.467]",0


In [16]:
pressure_oil_heatmap_pg = pressure_oil_heatmap.copy()
pressure_impact_pg = pressure_impact.copy()
temperature_impact_pg = temperature_impact.copy()

def interval_to_str(interval):
    if pd.isna(interval):
        return None
    left = interval.left
    right = interval.right
    return f"{round(float(left), 2)} – {round(float(right), 2)}"

pressure_oil_heatmap_pg["pressure_bin_str"] = pressure_oil_heatmap_pg["pressure_bin"].apply(interval_to_str)
pressure_oil_heatmap_pg["oil_bin_str"] = pressure_oil_heatmap_pg["oil_bin"].apply(interval_to_str)

pressure_oil_heatmap_pg = pressure_oil_heatmap_pg[["pressure_bin_str", "oil_bin_str", "records_count"]]

if "pressure_bin" in pressure_impact_pg.columns:
    pressure_impact_pg["pressure_bin_str"] = pressure_impact_pg["pressure_bin"].apply(interval_to_str)
    pressure_impact_pg = pressure_impact_pg[["pressure_bin_str", "avg_oil_ton", "avg_temperature", "rows"]]

if "temperature_bin" in temperature_impact_pg.columns:
    temperature_impact_pg["temperature_bin_str"] = temperature_impact_pg["temperature_bin"].apply(interval_to_str)
    temperature_impact_pg = temperature_impact_pg[["temperature_bin_str", "avg_oil_ton", "avg_pressure", "rows"]]

In [17]:
engine = get_pg_engine()

mart_production_daily.to_sql(
    "mart_production_daily",
    engine,
    if_exists="replace",
    index=False
)

mart_well_kpi.to_sql(
    "mart_well_kpi",
    engine,
    if_exists="replace",
    index=False
)

pressure_oil_heatmap_pg.to_sql(
    "mart_pressure_oil_heatmap",
    engine,
    if_exists="replace",
    index=False
)

pressure_impact_pg.to_sql(
    "mart_pressure_impact",
    engine,
    if_exists="replace",
    index=False
)

temperature_impact_pg.to_sql(
    "mart_temperature_impact",
    engine,
    if_exists="replace",
    index=False
)

5

In [18]:
pd.read_sql("SELECT * FROM mart_production_daily LIMIT 5", engine)

,date,total_oil_ton,total_gas_m3,total_water_m3,total_energy_kwh,total_downtime_hours,avg_temperature,avg_pressure,active_wells,downtime_pct
0,2025-10-01,717.5,197170.0,631.1,26730.0,27.7,84.625,115.425,5,23.083333
1,2025-10-02,717.3,197200.0,630.5,26735.0,28.0,84.625,115.475,5,23.333333
2,2025-10-03,719.2,197350.0,630.8,26770.0,27.8,84.875,115.500,5,23.166667
3,2025-10-04,722.9,197810.0,627.4,26875.0,26.7,84.375,116.000,5,22.250000
4,2025-10-05,721.0,197620.0,630.1,26810.0,27.5,84.675,115.725,5,22.916667


In [19]:
pd.read_sql("SELECT * FROM mart_well_kpi LIMIT 5", engine)

,well_id,total_oil_ton,avg_oil_ton,total_downtime_hours,avg_prod_temperature,avg_prod_pressure,days_count,downtime_pct,avg_telemetry_temperature,avg_telemetry_pressure,avg_oil_flow_rate,avg_vibration,well_rank_by_avg_oil
0,1,6394.5,213.150000,14.5,88.206667,120.543333,30,2.013889,88.027264,122.016986,8.801222,1.741514,1.0
1,5,5952.9,198.430000,13.1,86.800000,119.426667,30,1.819444,86.521034,119.422141,8.201379,2.258319,2.0
2,2,5574.5,185.816667,21.8,84.486667,115.306667,30,3.027778,84.328472,115.412069,7.501347,1.743972,3.0
3,3,3650.9,121.696667,58.4,79.433333,107.160000,30,8.111111,NaN,NaN,NaN,NaN,4.0
4,4,0.0,0.000000,720.0,NaN,NaN,30,100.000000,NaN,NaN,NaN,NaN,5.0


In [20]:
pd.read_sql("SELECT * FROM mart_pressure_oil_heatmap LIMIT 5", engine)

,pressure_bin_str,oil_bin_str,records_count
0,106.39 – 108.92,-0.21 – 35.87,0
1,106.39 – 108.92,35.87 – 71.73,0
2,106.39 – 108.92,71.73 – 107.6,0
3,106.39 – 108.92,107.6 – 143.47,30
4,106.39 – 108.92,143.47 – 179.33,0


In [21]:
analysis_conclusions = {
    "best_well_by_avg_oil": top_wells.iloc[0]["well_id"],
    "best_well_avg_oil": top_wells.iloc[0]["avg_oil_ton"],
    "worst_well_by_avg_oil": worst_wells.iloc[0]["well_id"],
    "worst_well_avg_oil": worst_wells.iloc[0]["avg_oil_ton"],
    "highest_downtime_well": highest_downtime.iloc[0]["well_id"],
    "highest_downtime_pct": highest_downtime.iloc[0]["downtime_pct"],
    "correlation_oil_pressure": corr_matrix.loc["oil_ton", "pressure"],
    "correlation_oil_temperature": corr_matrix.loc["oil_ton", "temperature"],
}

pd.DataFrame(list(analysis_conclusions.items()), columns=["metric", "value"])

,metric,value
0,best_well_by_avg_oil,1.000000
1,best_well_avg_oil,213.150000
2,worst_well_by_avg_oil,4.000000
3,worst_well_avg_oil,0.000000
4,highest_downtime_well,4.000000
5,highest_downtime_pct,100.000000
6,correlation_oil_pressure,0.987107
7,correlation_oil_temperature,0.985105


В рамках аналитики добычи были построены витрины:
- общая добыча по дням;
- KPI по скважинам;
- зависимость дебита от давления и температуры.

Результаты выгружены в PostgreSQL для последующей визуализации в Apache Superset.